In [ ]:
#| hide
# from nbmcp.core import *

# nbdev_mcp

> nbdev MCP server (multi-project, rich output, workflow-aware) — Python 3.12+

### Features

- Works from anywhere: select a project by path or alias, or pass project= per call.
  
- Multi-project support: bookmarks (alias -> path), discovery via `NBDEV_PROJECTS` or common roots.
  
- Tools: `nbdev_prepare`, `nbdev_export`, `nbdev_test`, `pytest`, `ensure_env`, `export_env` (mamba/conda).
  
- **Notebook editing**: `check_if_generated`, `find_source_notebook`, edit/read/add notebook cells.
  
- **Workflow guidance**: Prompts that instruct your choice of opensource models / prioprietary (e.g. Codex, Claude) to use nbdev properly (edit notebooks, not .py files).
  
- `Resources`: project summary, tree, settings.ini, env file, read file (safe, read-only).
  
- `Prompts`: nbdev_workflow_philosophy (CRITICAL), `nbdev_howto`, `module_scaffold`.
  
- `Transports`: stdio (for desktop clients), streamable-http (built-in default HTTP), or HTTP via Uvicorn for custom host/port/path.
  
- Rich-formatted "pretty" output for UI display; never prints to stdout in stdio mode.

### Philosophy

In nbdev, `.ipynb` notebooks are _source code_. Python `.py` files are _generated_.

This MCP guides LLMs to follow this philosophy by providing tools to:

1. Check if files are generated (`check_if_generated`)

2. Find source notebooks (`find_source_notebook`)
   
3. Edit notebooks instead of `.py` files (`edit_notebook_cell`)
   
4. Always run `nbdev_export` after editing


## Integrations

### VS Code

#### Command Dialog (<kbd>Cmd</kbd> + <kbd>Shift</kbd> + <kbd>P</kbd>)

Press <kbd>Cmd</kbd> + <kbd>Shift</kbd> + <kbd>P</kbd> to open up the command dialog in VS Code. Then start typing `'MCP'` to find MCP related commands.

You are looking for `'MCP: Add Server...'`

#### Config File

First you need to find where your Visual Studio Code application support is installed. 

For example, if you are on macOS this might be

```sh
~/Library/Application Support/Code/
```

What you are looking for is a `mcp.json` file. This might already exist if you've installed MCPs from VSCode Extensions. 

If not you can create the `mcp.json` configuration file scoped for the user in the directory:

```sh
$ touch ~/Library/Application Support/Code/User/mcp.json
```

Then in `mcp.json`

```json
# ~/Library/Application Support/Code/User/mcp.json
{
    "servers": {
        // ... other servers
        "nbdev-mcp": {
            "type": "stdio",
            "command": "mamba",
            "args": [
                "run",
                "-n",
                "core",  // if you want to point to a specific mamba / conda environment
                "python",
                "-m",
                "nbdev_mcp",
                "--transport",
                "stdio" // "streamable-http"
            ]
        }
    }
}
```

### Claude Code

#### Config File

Inside `~/.claude.json`

```json
{
    // ... add globally
    "mcpServers": { 
        "prompt_toolkit": {
          "type": "stdio",
          "command": "/Path/to/mamba/envs/env_name/bin/python",
          "args": [
            "-m",
            "nbdev_mcp",
            "--transport",
            "stdio"
          ],
          "env": {}
        }
      },
    
    // alternatively under a specific project
    "projects": {
      "Path/to/project": { // added automatically by claude
        // ...
        "mcpServers": { 
          // same as above
        }
        // ...
      }
    }
}
```

#### Command Line

```sh
PYTHON="python3" # PYTHON="~/{mamba|conda|micromamba}/envs/{env_name}/bin/python"

MCP_SCRIPT="~/MCPs/nbdev_mcp" # path to script

# Remove existing configuration if present
claude mcp remove nbdev 2>/dev/null || true

# Add the MCP server - using user scope so it's available everywhere
claude mcp add --scope user --transport stdio nbdev -- \
    "$PYTHON" -m "$MCP_SCRIPT" --transport stdio
```

### Codex

#### Config File

Under you codex config, e.g. `~/.codex/config.toml`

```ini
# https://developers.openai.com/codex/local-config/

# Default model to use
model = "gpt-5.1-codex-max"
model_provider = "openai"   # or e.g., "ollama" etc

# ...
[mcp_servers.mcp-nbdev]
command = "/Path/to/{mamba|conda|micromamba}/envs/{env_name}/bin/python"
args = ["-m", "nbdev_mcp", "--transport", "stdio"]
```

#### Command Line

```sh
codex mcp add mcp-nbdev \ 
    --env NBDEV_PROJECTS="~/dev:~/Projects" \
    /Path/to/{mamba|conda|micromamba}/envs/{env_name}/bin/python \
    -m nbdev_mcp --transport stdio
````

## Developer Guide

If you are new to using `nbdev` here are some useful pointers to get you started.

### Install nbmcp in Development mode

```sh
# make sure nbmcp package is installed in development mode
$ pip install -e .

# make changes under nbs/ directory
# ...

# compile to have changes apply to nbmcp
$ nbdev_prepare
```

## Usage

### Installation

Install latest from the GitHub [repository][repo]:

```sh
$ pip install git+https://github.com/s01st/nbdev_mcp.git
```

or from [conda][conda]

```sh
$ conda install -c s01st nbdev_mcp
```

or from [pypi][pypi]


```sh
$ pip install nbdev_mcp
```


[repo]: https://github.com/s01st/nbdev_mcp
[docs]: https://s01st.github.io/nbdev_mcp/
[pypi]: https://pypi.org/project/nbdev_mcp/
[conda]: https://anaconda.org/s01st/nbdev_mcp

### Documentation

Documentation can be found hosted on this GitHub [repository][repo]'s [pages][docs]. Additionally you can find package manager specific guidelines on [conda][conda] and [pypi][pypi] respectively.

[repo]: https://github.com/s01st/nbmcp
[docs]: https://s01st.github.io/nbmcp/
[pypi]: https://pypi.org/project/nbmcp/
[conda]: https://anaconda.org/s01st/nbmcp

## How to use

```sh
$ python -m nbdev_mcp --transport http --host 127.0.0.1 --port 8766 --path /mcp
```

```sh
(core) ➜  ~ python -m nbdev_mcp --transport http --host 127.0.0.1 --port 8766 --path /mcp
INFO:     Started server process [47867]
INFO:     Waiting for application startup.
[12/03/25 11:06:02] INFO     StreamableHTTP session manager started   streamable_http_manager.py:110
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8766 (Press CTRL+C to quit)
```